# SIV-WAM 手撕学习 · 02：训练阶段的 Flow Matching

本 Notebook 开始进入真正的训练阶段。这里暂时不调用 Wan Transformer，而是先手写训练最核心的数学机制：`FlowMatchScheduler`。

你需要自己完成的函数：

```text
sample timestep
set_timesteps / sigma schedule
add_noise
training_target
training_weight
step
```

生产参考文件：
`C:\Users\10752\Documents\ChatGPT\WAM\remote_snapshot_20260918\siv_wam\utils\scheduler.py`

本 Notebook 会故意留出关键代码空白。先根据公式自己写，再用测试验证。

## 本节训练主线

数据集输出的是 clean latent：

```text
clean latent x0
    + random noise ε
    + noise level σ
        ↓
noisy latent xσ
        ↓
模型预测 velocity
        ↓
与 target = ε - x0 比较
```

核心公式：

$x_\sigma = (1-\sigma)x_0 + \sigma\epsilon$

$v^* = \epsilon - x_0$

In [65]:
import math
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd()
REFERENCE_SCHEDULER = (
    PROJECT_ROOT / 'siv_wam' /
    'utils' / 'scheduler.py'
)
print('torch:', torch.__version__)
print('reference exists:', REFERENCE_SCHEDULER.exists())

torch: 2.9.0+cu126
reference exists: True


## 1. 先实现 timestep 采样

生产代码中的采样逻辑位于：`siv_wam/utils/utils.py::sample_timestep_id`。

要求：

- 返回长度为 `num_samples` 的整型 tensor；
- 每个值位于 `[0, num_train_timesteps-1]`；
- 可以通过 `min_timestep_bd/max_timestep_bd` 限制采样区间。

先自己填写下面的函数，不要查看生产实现。

In [ ]:
def sample_timestep_id_todo(
    num_samples=1,
    min_timestep_bd=0.0,
    max_timestep_bd=1.0,
    num_train_timesteps=1000,
):
    # 1：均匀采样 [min_timestep_bd, max_timestep_bd] 中的 u
    u = torch.randn(size=[num_samples]) # 取离散变量
    u = u * (max_timestep_bd - min_timestep_bd) + min_timestep_bd # 映射
    # 2：将 u 映射到离散 timestep id
    timestep_id = (u * num_train_timesteps).clamp(min=min_timestep_bd*num_train_timesteps, max=num_train_timesteps - 1)
    # 3：返回 torch.int64 tensor
    return timestep_id.to(torch.int64)

In [67]:
# 完成上一个 cell 后运行本测试
torch.manual_seed(0)
t_ids = sample_timestep_id_todo(num_samples=1000, num_train_timesteps=1000)
print(t_ids.shape, t_ids.dtype, int(t_ids.min()), int(t_ids.max()))
assert t_ids.shape == (1000,)
assert t_ids.dtype == torch.int64
assert 0 <= int(t_ids.min())
assert int(t_ids.max()) < 1000

late_ids = sample_timestep_id_todo(1000, 0.5, 1.0, 1000)
assert int(late_ids.min()) >= 500

torch.Size([1000]) torch.int64 0 999


## 2. 实现 sigma schedule

参考生产代码的 `FlowMatchScheduler.set_timesteps()`，你需要理解三层关系：

```text
初始 sigma 网格
    ↓ shift
最终 sigmas
    ↓ × num_train_timesteps
timesteps
```

当前训练配置会使用：

```text
num_train_timesteps = 1000
sigma_min = 0
sigma_max = 1
extra_one_step = True
```

提示：`shift` 会改变 sigma 的分布，不是简单的加法。生产代码使用：

```python
sigmas = shift * sigmas / (1 + (shift - 1) * sigmas)
```

In [68]:
class FlowMatchSchedulerTodo:
    def __init__(
        self,
        num_train_timesteps=1000,
        shift=5.0,
        sigma_min=0.0,
        sigma_max=1.0,
        extra_one_step=True,
    ):
        self.num_train_timesteps = num_train_timesteps
        self.shift = shift # shift=1：均匀的 sigma schedule shift > 1：让 schedule 更偏向某些噪声区域
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.extra_one_step = extra_one_step
        self.sigmas = None
        self.timesteps = None
        self.set_timesteps(num_train_timesteps, training=True)

    def set_timesteps(self, num_inference_steps=100, training=False): # num_inference_steps 推理时要进行多少次 denoising 更新。
        # TODO：实现初始 sigma 网格
        # TODO：处理 extra_one_step
        sigma_start = self.sigma_max
        if self.extra_one_step:
            # 不是让 sigmas 最后一个一定为 0，而是让整个去噪过程拥有更细的步长；最后一步到 0 会在 step() 中单独处理。?
            sigmas = torch.linspace(sigma_start, self.sigma_min, num_inference_steps + 1)[:-1] # 去掉最后一个
        else:
            sigmas = torch.linspace(sigma_start, self.sigma_min, num_inference_steps)
        # TODO：应用 shift: shift * sigmas / (1 +shift - 1) * sigmas 非线性变换 
        shift_sigmas = self.shift * sigmas / (1 + (self.shift - 1) * sigmas)
        self.sigmas = shift_sigmas
        # TODO：构造 self.timesteps
        self.timesteps = self.sigmas * self.num_train_timesteps # sigma 是连续噪声强度 timestep 是 scheduler 使用的离散时间坐标
        # TODO：training=True 时保存训练权重所需的信息
        self.training = training # 是否额外计算训练阶段使用的 timestep loss 权重。
        # raise NotImplementedError('请先完成 set_timesteps')

In [69]:
# 完成 set_timesteps 后运行
scheduler = FlowMatchSchedulerTodo(num_train_timesteps=1000, shift=5.0)
print('sigmas:', scheduler.sigmas.shape, scheduler.sigmas[:3], scheduler.sigmas[-3:])
print('timesteps:', scheduler.timesteps.shape, scheduler.timesteps[:3], scheduler.timesteps[-3:])
assert scheduler.sigmas.ndim == 1
assert scheduler.timesteps.shape == scheduler.sigmas.shape
assert torch.all(scheduler.sigmas >= 0)
assert torch.all(scheduler.sigmas <= 1)
assert torch.all(scheduler.sigmas[:-1] >= scheduler.sigmas[1:])

sigmas: torch.Size([1000]) tensor([1.0000, 0.9998, 0.9996]) tensor([0.0148, 0.0099, 0.0050])
timesteps: torch.Size([1000]) tensor([1000.0000,  999.7998,  999.5994]) tensor([14.8221,  9.9206,  4.9801])


## 3. 实现 `add_noise()`

生产代码的核心是：

```python
sample = (1 - sigma) * original_samples + sigma * noise
```

注意两个实现问题：

1. timestep 需要映射到最近的 sigma；
2. timestep 要沿 latent 的时间维广播。

对于 video/mask，时间维是 `t_dim=2`，因为 tensor 布局是 `[B,C,F,H,W]`。

In [72]:
def add_noise_todo(scheduler, original_samples, noise, timestep, t_dim=2):
    # TODO 1：找到 timestep 对应的最近 sigma index
    t = torch.as_tensor(timestep).reshape(-1)
    distance = (scheduler.timesteps[:,None] - t[None,:]).abs() # None 相当于 unsqueeze() 操作
    sigma_index = torch.argmin(distance, dim=0) # 每行中最小的，就是最近的
    # TODO 2：把 sigma reshape 成可以沿 t_dim 广播的形状 [F] -> [B,C,F,H,W]
    view_shape = [1] * original_samples.ndim # 5
    view_shape[t_dim] = scheduler.sigmas[sigma_index].shape[0]
    sigma = scheduler.sigmas[sigma_index].reshape(view_shape)
    # TODO 3：返回 (1-sigma)*original + sigma*noise
    return (1-sigma)*original_samples +sigma*noise


In [74]:
# 完成 add_noise_todo 后运行
clean = torch.ones(2, 3, 4, 2, 2)
noise = torch.zeros_like(clean)
sigma_zero_t = scheduler.timesteps[torch.argmin(scheduler.sigmas.abs())]
sigma_one_t = scheduler.timesteps[torch.argmax(scheduler.sigmas)]
x_zero = add_noise_todo(scheduler, clean, noise, sigma_zero_t, t_dim=2)
x_one = add_noise_todo(scheduler, clean, noise, sigma_one_t, t_dim=2)
print('sigma=0 result mean:', x_zero.mean().item())
print('sigma=max result mean:', x_one.mean().item())
min_sigma_index = torch.argmin(scheduler.sigmas.abs())
max_sigma_index = torch.argmax(scheduler.sigmas)

sigma_min_actual = scheduler.sigmas[min_sigma_index].to(
    device=clean.device,
    dtype=clean.dtype,
)

sigma_max_actual = scheduler.sigmas[max_sigma_index].to(
    device=clean.device,
    dtype=clean.dtype,
)

sigma_zero_t = scheduler.timesteps[min_sigma_index]
sigma_one_t = scheduler.timesteps[max_sigma_index]

x_min_sigma = add_noise_todo(
    scheduler,
    clean,
    noise,
    sigma_zero_t,
    t_dim=2,
)

x_max_sigma = add_noise_todo(
    scheduler,
    clean,
    noise,
    sigma_one_t,
    t_dim=2,
)

expected_min = (
    (1 - sigma_min_actual) * clean
    + sigma_min_actual * noise
)

expected_max = (
    (1 - sigma_max_actual) * clean
    + sigma_max_actual * noise
)

assert x_min_sigma.shape == clean.shape
assert x_max_sigma.shape == clean.shape

assert torch.allclose(
    x_min_sigma,
    expected_min,
    atol=1e-5,
)

assert torch.allclose(
    x_max_sigma,
    expected_max,
    atol=1e-5,
)

print("min sigma:", sigma_min_actual.item())
print("max sigma:", sigma_max_actual.item())
print("add_noise test passed")

sigma=0 result mean: 0.9950199127197266
sigma=max result mean: 0.0
min sigma: 0.004980080295354128
max sigma: 1.0
add_noise test passed


## 4. 实现 `training_target()`

生产代码只有一行：

```python
target = noise - sample
```

这里的 `sample` 是 clean latent，不是 `noisy_latent`。这是本项目训练目标最容易写错的地方。

In [ ]:
def training_target_todo(clean, noise):
    # 返回 flow-matching velocity target
    return noise - clean

clean_small = torch.tensor([1.0, 2.0])
noise_small = torch.tensor([3.0, -2.0])
target_small = training_target_todo(clean_small, noise_small)
print(target_small)
assert torch.equal(target_small, torch.tensor([2.0, -4.0]))

tensor([ 2., -4.])


## 5. 实现 inference `step()`

训练时模型学习 velocity；推理时 scheduler 根据 velocity 更新 sample：

```python
prev_sample = sample + model_output * (sigma_next - sigma)
```

这一步把训练目标和推理过程连接起来。你需要让 `step()` 找到当前 timestep 的 sigma，并选择下一个 sigma。

In [ ]:
def step_todo(scheduler, model_output, timestep, sample):
    # TODO 1：找到当前 timestep 对应的 sigma
    t = torch.as_tensor(timestep).reshape(-1)
    distance = (scheduler.timesteps[:,None] - t[None,:]).abs()
    sigma_idx = torch.argmin(distance, dim=0)
    sigma = scheduler.sigmas[sigma_idx]
    # TODO 2：找到 sigma_next；最后一步使用 0
    sigma_next = scheduler.sigmas[sigma_idx+1]
    # TODO 3：执行 sample + velocity * (sigma_next - sigma)
    return sample + model_output * (sigma_next - sigma)

## 本 Notebook 的完成标准

完成后必须通过以下逻辑检查：

```text
sigma=0      → noisy == clean
sigma=max    → noisy ≈ noise
target       = noise - clean
add_noise    不改变 tensor shape
step         能沿 sigma 方向更新 sample
video/mask   使用 latent scheduler
action       后续使用另一套 scheduler
```

请先只完成 `sample_timestep_id_todo`、`set_timesteps`、`add_noise_todo` 和 `training_target_todo`。`step_todo` 可以最后完成。遇到错误时，把对应 cell 的代码和 traceback 发给我，我会只提示关键思路，不直接替你填空。